In [1]:
import pandas as pd
import json

  Example of the json record we're reading
  
  ```json
  "956d8f9f18208168bb30bbac9299bb59": {
    "aiResults": [
      {
        "modelName": "speciesnet/PyTorch/v4.0.1a",
        "runDate": "2025-05-24",
        "confBlank": 0.99,
        "confHuman": 0.0,
        "confAnimal": 0.01
      }
    ]
  },
  ```

In [2]:
with open('../mongodb_formatted_detections.json', 'r') as f:
    data = json.load(f)

df = pd.DataFrame(data)
bins = pd.interval_range(start=0, end=1, freq=0.05, closed='left')

records = []
for key, value in data.items():
    for result in value.get('aiResults', []):
        record = {'id': key}
        record.update(result)
        records.append(record)

df = pd.DataFrame(records)
df.head()

,id,modelName,runDate,confBlank,confHuman,confAnimal
0,c112813a5f3b9cec26f95fad982b8d09,speciesnet/PyTorch/v4.0.1a,2025-07-07,1.00,0.00,0.0
1,0647380f2d59692f5b2b642312844e9f,speciesnet/PyTorch/v4.0.1a,2025-07-07,0.79,0.21,0.0
2,0db73c6c1efb4968c04a47e418ebeefb,speciesnet/PyTorch/v4.0.1a,2025-07-07,0.36,0.64,0.0
3,31fc53de29056b4dd8bc7b1804617f00,speciesnet/PyTorch/v4.0.1a,2025-07-07,0.27,0.73,0.0
4,14664d764836c5fd9a38284dc6103527,speciesnet/PyTorch/v4.0.1a,2025-07-07,0.29,0.71,0.0


In [3]:

df['conf_bin'] = pd.cut(df['confAnimal'], bins)
counts = df['conf_bin'].value_counts().sort_index()


In [4]:

# When printing, round the bin edges to 2 decimals for display
rounded_counts = counts.copy()
rounded_counts.index = pd.IntervalIndex.from_tuples(
    [(round(i.left, 2), round(i.right, 2)) for i in counts.index],
    closed='left'
)
display(rounded_counts)

[0.0, 0.05)    278929
[0.05, 0.1)     23449
[0.1, 0.15)     12502
[0.15, 0.2)      5342
[0.2, 0.25)      5276
[0.25, 0.3)      4968
[0.3, 0.35)      3449
[0.35, 0.4)      2553
[0.4, 0.45)      2947
[0.45, 0.5)      2966
[0.5, 0.55)      2940
[0.55, 0.6)      3697
[0.6, 0.65)      2804
[0.65, 0.7)      4717
[0.7, 0.75)      3822
[0.75, 0.8)      6175
[0.8, 0.85)     11156
[0.85, 0.9)     12484
[0.9, 0.95)     31915
[0.95, 1.0)     10099
Name: count, dtype: int64

In [ ]:
import ipywidgets as widgets

from IPython.display import display

def show_counts(threshold):
    above = (counts.index.right > threshold).sum()
    below = (counts.index.right <= threshold).sum()
    print(f"Threshold: {threshold:.2f}")
    print(f"Bins with upper edge > threshold: {above}")
    print(f"Bins with upper edge <= threshold: {below}")

slider = widgets.FloatSlider(
    value=0.5,
    min=0.0,
    max=1.0,
    step=0.01,
    description='Threshold:',
    continuous_update=False
)

widgets.interact(show_counts, threshold=slider)